# Ask Sage API -- Python Notebook

A hands-on guide to the Ask Sage API covering both **raw REST** (`requests` library) and the **`asksageclient`** Python package. Each endpoint is demonstrated with both approaches so you can choose what fits your workflow.

The Ask Sage API is split into two base URLs:
- **User API** (`api.asksage.ai/user/`) -- User management, authentication, and dataset operations
- **Server API** (`api.asksage.ai/server/`) -- Core AI operations, model queries, training, and MCP management

**Resources:**
- [Server API Swagger Docs](https://app.swaggerhub.com/apis-docs/asksageinc/ask-sage_server_api/1.56)
- [User API Swagger Docs](https://app.swaggerhub.com/apis-docs/asksageinc/ask-sage_user_api/1.21)
- [asksageclient on PyPI](https://pypi.org/project/asksageclient/)
- [Ask Sage Documentation](https://docs.asksage.ai)

---

## Table of Contents

**Setup**
1. [Setup & Installation](#1-setup--installation)
2. [Configuration](#2-configuration)

**User API Endpoints** (`api.asksage.ai/user/`)
3. [Authentication](#3-authentication)
4. [Add Dataset](#4-add-dataset)
5. [Get User Logs](#5-get-user-logs)
6. [Get Chats](#6-get-chats)

**Server API Endpoints** (`api.asksage.ai/server/`)
7. [Get Models](#7-get-models)
8. [Get Personas](#8-get-personas)
9. [Get Datasets](#9-get-datasets)
10. [Query](#10-query)
11. [Query with File](#11-query-with-file)
12. [Follow-Up Questions](#12-follow-up-questions)
13. [Train with File](#13-train-with-file)
14. [Count Monthly Tokens](#14-count-monthly-tokens)
15. [Add MCP Server](#15-add-mcp-server)

**Agent Builder**
16. [List Agents & Execute Agent](#16-agent-builder)

17. [Tips & Best Practices](#17-tips--best-practices)

## 1. Setup & Installation

Install the required packages. Only `requests` and `asksageclient` are required; `pandas` is optional for nicer table display.

In [ ]:
!pip install -q requests asksageclient pandas

## 2. Configuration

Set your credentials via environment variables or replace the placeholder strings below.

| Variable | Description |
|----------|-------------|
| `ASKSAGE_EMAIL` | Your Ask Sage account email |
| `ASKSAGE_API_KEY` | Your static API key (from Account Settings) |

In [ ]:
import os
import json
import requests

# Credentials -- set via environment variables or replace the strings below
EMAIL = os.getenv("ASKSAGE_EMAIL", "your_email@example.com")
API_KEY = os.getenv("ASKSAGE_API_KEY", "your_api_key_here")

# Base URLs
USER_BASE_URL = "https://api.asksage.ai/user"
SERVER_BASE_URL = "https://api.asksage.ai/server"

# Optional: pandas for table display
try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

print(f"Email: {EMAIL}")
print(f"Server URL: {SERVER_BASE_URL}")
print(f"Pandas available: {HAS_PANDAS}")

---

# User API Endpoints

> **Base URL:** `https://api.asksage.ai/user/`

The User API handles authentication, dataset creation, and user management.

---

## 3. Authentication

**`POST /user/get-token-with-api-key`** -- Exchange your API key for a 24-hour access token.

Ask Sage uses the `x-access-tokens` header for authentication. You can use either:

- **Static API key** -- use directly as the token value (no expiry)
- **24-hour access token** -- exchange your API key for a short-lived token via `/get-token-with-api-key`

### Raw REST Approach

In [ ]:
# Get a 24-hour access token
auth_response = requests.post(
    f"{USER_BASE_URL}/get-token-with-api-key",
    json={"email": EMAIL, "api_key": API_KEY}
)

auth_data = auth_response.json()
print(f"Status: {auth_response.status_code}")
print(f"Response: {auth_data.get('response', 'N/A')}")

# Store the token and build reusable headers
TOKEN = auth_data.get("access_token", API_KEY)  # fallback to static key
HEADERS = {
    "x-access-tokens": TOKEN,
    "Content-Type": "application/json"
}

print(f"Token obtained: {'Yes' if 'access_token' in auth_data else 'Using static API key'}")

### asksageclient Approach

The client handles token management internally.

In [ ]:
from asksageclient import AskSageClient

client = AskSageClient(email=EMAIL, api_key=API_KEY)
print("Client initialized successfully")

---

## 4. Add Dataset

**`POST /user/add-dataset`** -- Create a new dataset in your account. Datasets serve as RAG knowledge bases that the AI can reference.

> **Naming rules:** Dataset names must be alphanumeric only (no spaces). Prefix with `CUI-` for Controlled Unclassified Information.

| Parameter | Type | Description |
|-----------|------|-------------|
| `dataset` | string | Name of the dataset to create (required, alphanumeric, no spaces) |

### Raw REST

In [ ]:
# Note: this uses the USER API base URL, not the server URL
resp = requests.post(
    f"{USER_BASE_URL}/add-dataset",
    headers=HEADERS,
    json={"dataset": "MyDemoDataset"}
)

dataset_data = resp.json()
print(f"Status: {resp.status_code}")
print(json.dumps(dataset_data, indent=2))

### asksageclient

In [ ]:
result = client.add_dataset("MyDemoDataset")
print(json.dumps(result, indent=2))

---

## 5. Get User Logs

**`POST /user/get-user-logs`** -- Retrieve your latest prompts and completions. Returns an array of log objects with timestamps, prompts, and AI responses.

### Raw REST

In [ ]:
resp = requests.post(
    f"{USER_BASE_URL}/get-user-logs",
    headers=HEADERS,
    json={}
)

logs_data = resp.json()
print(f"Status: {resp.status_code}")

if HAS_PANDAS and "response" in logs_data and isinstance(logs_data["response"], list):
    df = pd.DataFrame(logs_data["response"])
    if not df.empty:
        cols = [c for c in ["date_time", "prompt", "completion"] if c in df.columns]
        display(df[cols].head(10))
    else:
        print("No logs found.")
else:
    print(json.dumps(logs_data, indent=2))

### asksageclient

In [ ]:
result = client.get_user_logs()

if HAS_PANDAS and "response" in result and isinstance(result["response"], list):
    df = pd.DataFrame(result["response"])
    if not df.empty:
        cols = [c for c in ["date_time", "prompt", "completion"] if c in df.columns]
        display(df[cols].head(10))
else:
    print(json.dumps(result, indent=2))

---

## 6. Get Chats

**`POST /user/get-chats`** -- Retrieve all chat sessions for the authenticated user.

### Raw REST

In [ ]:
resp = requests.post(
    f"{USER_BASE_URL}/get-chats",
    headers=HEADERS,
    json={}
)

chats_data = resp.json()
print(f"Status: {resp.status_code}")
print(json.dumps(chats_data, indent=2))

### asksageclient

In [ ]:
result = client.get_chats()
print(json.dumps(result, indent=2))

---

# Server API Endpoints

> **Base URL:** `https://api.asksage.ai/server/`

The Server API handles core AI operations including model queries, training, MCP server management, and agent execution.

---

## 7. Get Models

**`POST /server/get-models`** -- Returns a list of all AI models available to your tenant.

### Raw REST

In [ ]:
resp = requests.post(f"{SERVER_BASE_URL}/get-models", headers=HEADERS)
models_data = resp.json()

print(f"Status: {resp.status_code}")

if HAS_PANDAS and "response" in models_data:
    model_list = models_data["response"].get("data", [])
    df = pd.DataFrame(model_list)
    if not df.empty:
        display(df[["id", "name", "owned_by"]].head(20))
        print(f"\nTotal models: {len(model_list)}")
else:
    print(json.dumps(models_data, indent=2))

### asksageclient

In [ ]:
models = client.get_models()

if HAS_PANDAS:
    model_list = models.get("response", {}).get("data", [])
    display(pd.DataFrame(model_list)[["id", "name", "owned_by"]].head(20))
else:
    print(json.dumps(models, indent=2))

---

## 8. Get Personas

**`POST /server/get-personas`** -- Personas are pre-defined system prompts that shape the AI's behavior. Use the persona `id` in `/query` calls.

### Raw REST

In [ ]:
resp = requests.post(f"{SERVER_BASE_URL}/get-personas", headers=HEADERS)
personas_data = resp.json()

print(f"Status: {resp.status_code}")

if HAS_PANDAS and "response" in personas_data:
    personas = personas_data["response"]
    if isinstance(personas, list):
        df = pd.DataFrame(personas)
        display(df[["id", "name", "description"]].head(10) if "description" in df.columns else df.head(10))
else:
    print(json.dumps(personas_data, indent=2))

### asksageclient

In [ ]:
personas = client.get_personas()
print(json.dumps(personas, indent=2))

---

## 9. Get Datasets

**`POST /server/get-datasets`** -- Returns available RAG knowledge bases. These contain trained content that the AI references when answering queries.

### Raw REST

In [ ]:
resp = requests.post(f"{SERVER_BASE_URL}/get-datasets", headers=HEADERS)
datasets_data = resp.json()

print(f"Status: {resp.status_code}")
print(json.dumps(datasets_data, indent=2))

### asksageclient

In [ ]:
datasets = client.get_datasets()
print(json.dumps(datasets, indent=2))

---

## 10. Query

**`POST /server/query`** -- The main endpoint for generating AI completions.

**Key parameters:**

| Parameter | Type | Description |
|-----------|------|-------------|
| `message` | string/array | Your query (required) |
| `model` | string | AI model to use (default: `openai_gpt`) |
| `persona` | integer | Persona ID (default: 1) |
| `dataset` | string/array | Dataset(s) to query -- `"all"`, `"none"`, or specific names |
| `temperature` | number | Randomness 0.0-1.0 (default: 0.0) |
| `limit_references` | integer | Max knowledge base references (0 = no embeddings) |
| `live` | integer | Web search: 0=off, 1=Google, 2=Google+crawl |
| `system_prompt` | string | Custom system prompt (overrides default) |

### Basic Query -- Raw REST

In [ ]:
resp = requests.post(
    f"{SERVER_BASE_URL}/query",
    headers=HEADERS,
    json={
        "message": "What is Ask Sage?",
        "model": "gpt-4o-mini",
        "temperature": 0.0
    }
)

query_data = resp.json()
print(f"Status: {resp.status_code}\n")
print("AI Response:")
print(query_data.get("message", "No response"))

### Basic Query -- asksageclient

In [ ]:
result = client.query("What is Ask Sage?", model="gpt-4o-mini")
print(result.get("message", "No response"))

### Advanced Query -- Raw REST

Using persona, dataset, live search, and custom system prompt.

In [ ]:
resp = requests.post(
    f"{SERVER_BASE_URL}/query",
    headers=HEADERS,
    json={
        "message": "What are the latest cybersecurity threats?",
        "model": "gpt-4o-mini",
        "persona": 1,
        "dataset": "none",
        "temperature": 0.7,
        "limit_references": 3,
        "live": 1,
        "system_prompt": "You are a cybersecurity analyst. Be concise and specific."
    }
)

adv_data = resp.json()
print(f"Status: {resp.status_code}\n")
print("AI Response:")
print(adv_data.get("message", "No response"))

refs = adv_data.get("references", [])
if refs:
    print(f"\nReferences ({len(refs)}):")
    for ref in refs[:3]:
        print(f"  - {ref}")

### Advanced Query -- asksageclient

In [ ]:
result = client.query(
    message="What are the latest cybersecurity threats?",
    model="gpt-4o-mini",
    persona=1,
    dataset="none",
    temperature=0.7,
    limit_references=3,
    live=1,
    system_prompt="You are a cybersecurity analyst. Be concise and specific."
)

print(result.get("message", "No response"))

---

## 11. Query with File

**`POST /server/query_with_file`** -- Query the AI with file attachments for context-aware responses. All optional parameters from `/query` are also available here.

| Parameter | Type | Description |
|-----------|------|-------------|
| `message` | string | Your query about the file(s) (required) |
| `file` | string/array | File path(s) to include (required) |
| `model` | string | AI model to use |

### Raw REST

In [ ]:
resp = requests.post(
    f"{SERVER_BASE_URL}/query_with_file",
    headers=HEADERS,
    json={
        "file": "document.pdf",
        "message": "Summarize this document",
        "model": "gpt-4o-mini"
    }
)

qf_data = resp.json()
print(f"Status: {resp.status_code}\n")
print("AI Response:")
print(qf_data.get("message", "No response"))

### asksageclient

In [ ]:
result = client.query_with_file(
    file="document.pdf",
    message="Summarize this document",
    model="gpt-4o-mini"
)
print(result.get("message", "No response"))

---

## 12. Follow-Up Questions

**`POST /server/follow_up_questions`** -- Given conversation history, returns AI-suggested follow-up questions.

### Raw REST

In [ ]:
resp = requests.post(
    f"{SERVER_BASE_URL}/follow_up_questions",
    headers=HEADERS,
    json={
        "message": [
            {"user": "me", "message": "What is Ask Sage?"},
            {"user": "ai", "message": "Ask Sage is an AI platform that provides secure access to large language models."}
        ],
        "model": "gpt-4o-mini",
        "dataset": "none"
    }
)

followup_data = resp.json()
print(f"Status: {resp.status_code}")
print(json.dumps(followup_data, indent=2))

### asksageclient

In [ ]:
result = client.follow_up_questions(
    message=[
        {"user": "me", "message": "What is Ask Sage?"},
        {"user": "ai", "message": "Ask Sage is an AI platform that provides secure access to large language models."}
    ],
    model="gpt-4o-mini"
)
print(json.dumps(result, indent=2))

---

## 13. Train with File

**`POST /server/train-with-file`** -- Train a dataset using file content. Uses `multipart/form-data` for file upload.

**Note:** This consumes teach tokens from your monthly allocation.

| Parameter | Type | Description |
|-----------|------|-------------|
| `file` | binary | File to train from (required) |
| `dataset` | string | Target dataset name (optional) |

### Raw REST

In [ ]:
# Note: uses multipart/form-data, not JSON
file_path = "/path/to/your/file.pdf"  # <-- update this

with open(file_path, "rb") as f:
    resp = requests.post(
        f"{SERVER_BASE_URL}/train-with-file",
        headers={"x-access-tokens": TOKEN},
        files={"file": (os.path.basename(file_path), f, "application/pdf")},
        data={"dataset": "my_dataset"}
    )

train_data = resp.json()
print(f"Status: {resp.status_code}")
print(json.dumps(train_data, indent=2))

### asksageclient

In [ ]:
result = client.train_with_file(
    file="/path/to/your/file.pdf",  # <-- update this
    dataset="my_dataset"
)
print(json.dumps(result, indent=2))

---

## 14. Count Monthly Tokens

**`GET /server/count-monthly-tokens`** -- Returns the count of tokens used this month. Useful for monitoring usage and staying within allocation limits.

### Raw REST

In [ ]:
# Note: this is a GET request, not POST
resp = requests.get(
    f"{SERVER_BASE_URL}/count-monthly-tokens",
    headers={"x-access-tokens": TOKEN, "Accept": "application/json"}
)

token_count = resp.json()
print(f"Status: {resp.status_code}")
print(json.dumps(token_count, indent=2))

### asksageclient

In [ ]:
result = client.count_monthly_tokens()
print(json.dumps(result, indent=2))

---

## 15. Add MCP Server

**`POST /server/add-mcp-server`** -- Add a new MCP (Model Context Protocol) server for the authenticated user. MCP servers extend Ask Sage with external tool integrations.

| Parameter | Type | Description |
|-----------|------|-------------|
| `server_name` | string | Name of the MCP server (required) |
| `server_url` | string | URL of the MCP server (required) |
| `server_type` | string | Type of the MCP server (required) |
| `description` | string | Description of the MCP server (required) |
| `metadata_json` | string | Metadata as a JSON string (required) |

### Raw REST

In [ ]:
resp = requests.post(
    f"{SERVER_BASE_URL}/add-mcp-server",
    headers=HEADERS,
    json={
        "server_name": "My MCP Server",
        "server_url": "https://your-mcp-server.com",
        "server_type": "API",
        "description": "Custom MCP server for external tool integration",
        "metadata_json": json.dumps({"key": "value"})
    }
)

mcp_data = resp.json()
print(f"Status: {resp.status_code}")
print(json.dumps(mcp_data, indent=2))

---

# Agent Builder

> **Base URL:** `https://api.asksage.ai/server/`

The Agent Builder lets you create multi-step AI workflows with decision trees, loops, file processing, and more. These two endpoints let you list and execute agents programmatically.

> **Note:** Agent Builder must be enabled on your account. The `asksageclient` package may not yet have methods for these endpoints, so only raw REST is shown.

---

## 16. List Agents

**`GET /server/list-agents`** -- Returns all agents available to the authenticated user.

In [ ]:
resp = requests.get(
    f"{SERVER_BASE_URL}/list-agents",
    headers={"x-access-tokens": TOKEN}
)

agents_data = resp.json()
print(f"Status: {resp.status_code}")

if HAS_PANDAS and "response" in agents_data and isinstance(agents_data["response"], list):
    df = pd.DataFrame(agents_data["response"])
    if not df.empty:
        cols = [c for c in ["id", "name", "description", "agent_mode", "is_published"] if c in df.columns]
        display(df[cols])
    else:
        print("No agents found.")
else:
    print(json.dumps(agents_data, indent=2))

### Execute Agent

**`POST /server/execute-agent`** -- Runs an agent with your message. Supports streaming (SSE) and non-streaming modes.

| Parameter | Type | Description |
|-----------|------|-------------|
| `agent_id` | integer | Agent ID from `/list-agents` (required) |
| `message` | string | Your message/prompt (required) |
| `streaming` | boolean | Enable SSE streaming (default: true) |
| `variables` | object | Runtime variables for the agent |
| `conversation_history` | array | Previous conversation for context |

In [ ]:
# Replace AGENT_ID with an actual agent ID from the list above
AGENT_ID = 1  # <-- update this

resp = requests.post(
    f"{SERVER_BASE_URL}/execute-agent",
    headers=HEADERS,
    json={
        "agent_id": AGENT_ID,
        "message": "Summarize the key points.",
        "streaming": False
    }
)

exec_data = resp.json()
print(f"Status: {resp.status_code}")
print(f"Execution status: {exec_data.get('execution_status', 'N/A')}")
print(f"Duration: {exec_data.get('duration_ms', 'N/A')}ms")

agent_response = exec_data.get("response", {})
if isinstance(agent_response, dict):
    print(f"\nAgent Response:\n{agent_response.get('response', 'No response')}")
else:
    print(f"\nAgent Response:\n{agent_response}")

---

## 17. Tips & Best Practices

**Token Management**
- Use **static API keys** for development and testing
- Use **24-hour access tokens** for production applications -- they limit exposure if compromised

**Error Handling**
- Always check `response.status_code` before parsing JSON
- Handle `401` responses by refreshing your token
- Wrap API calls in try/except for network errors

**Performance**
- Set `limit_references=0` if you don't need RAG context (faster responses)
- Use `live=0` unless you specifically need web search results
- Use `temperature=0.0` for deterministic, factual responses

**Datasets**
- Use descriptive `context` values when training -- they help the AI find relevant content
- Dataset names must be alphanumeric (no spaces) -- create via the **User API** `/add-dataset`
- Train content into datasets via the **Server API** `/train-with-file`
- Monitor teach token usage with `GET /server/count-monthly-teach-tokens`
- Monitor query token usage with `GET /server/count-monthly-tokens`

**API Structure Reminder**
- **User API** (`api.asksage.ai/user/`) -- Authentication, dataset creation, user management, chat history
- **Server API** (`api.asksage.ai/server/`) -- AI queries, training, models, MCP, agents

**Further Reading**
- [Server API Swagger Docs](https://app.swaggerhub.com/apis-docs/asksageinc/ask-sage_server_api/1.56)
- [User API Swagger Docs](https://app.swaggerhub.com/apis-docs/asksageinc/ask-sage_user_api/1.21)
- [Ask Sage Documentation](https://docs.asksage.ai)
- [Agent Builder Guide](https://docs.asksage.ai/docs/agent-builder/)